# Ensemble Model

Train LightGBM, XGBoost, and CatBoost with a log1p target transform and blend predictions.

In [21]:
# Run this in terminal first if not installed
# pip install catboost
import catboost
print("CatBoost version:", catboost.__version__)

CatBoost version: 1.2.10


In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

print("Train:", train.shape)
print("Test: ", test.shape)

Train: (20886, 47)
Test:  (5300, 46)


In [23]:
def preprocess(df, train_medians=None, is_train=True):
    df = df.copy()

    df['generation_date'] = pd.to_datetime(df['generation_date'])
    df['gen_month'] = df['generation_date'].dt.month
    df['gen_day'] = df['generation_date'].dt.day
    df['gen_dow'] = df['generation_date'].dt.dayofweek
    df['is_monsoon'] = df['gen_month'].isin([5, 6, 7, 8, 9, 10, 11]).astype(int)
    df = df.drop(columns=['generation_date'])

    missing_indicator_cols = [
        'elevation_m', 'distance_to_river_m', 'drainage_index',
        'ndvi', 'ndwi', 'electricity', 'road_quality',
        'nearest_hospital_km', 'infrastructure_score'
    ]
    for col in missing_indicator_cols:
        if col in df.columns:
            df[f'{col}_is_missing'] = df[col].isnull().astype(int)

    log_cols = [
        'distance_to_river_m', 'population_density_per_km2',
        'rainfall_7d_mm', 'monthly_rainfall_mm',
        'nearest_hospital_km', 'nearest_evac_km',
        'inundation_area_sqm'
    ]
    for col in log_cols:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col].fillna(0))

    df['flood_susceptibility'] = (
        df['rainfall_7d_mm'].fillna(0) /
        (df['elevation_m'].fillna(df['elevation_m'].median()) + 1)
    )
    df['effective_drainage'] = (
        df['drainage_index'].fillna(0) *
        (1 - df['built_up_percent'].fillna(0) / 100)
    )
    df['rain_per_drainage'] = (
        df['rainfall_7d_mm'].fillna(0) /
        (df['drainage_index'].fillna(1) + 1)
    )
    df['river_elevation_risk'] = (
        1 / (df['distance_to_river_m'].fillna(9999) + 1) *
        1 / (df['elevation_m'].fillna(100) + 1)
    )

    num_cols = df.select_dtypes(include='number').columns.tolist()
    if is_train:
        train_medians = df[num_cols].median()
    df[num_cols] = df[num_cols].fillna(train_medians)

    cat_cols = df.select_dtypes(include='object').columns.tolist()
    cat_cols = [c for c in cat_cols if c not in ['record_id', 'flood_risk_score']]
    for col in cat_cols:
        df[col] = df[col].fillna('Missing')

    return df, train_medians

train_processed, medians = preprocess(train, is_train=True)
test_processed, _ = preprocess(test, train_medians=medians, is_train=False)

print("Processed train:", train_processed.shape)
print("Processed test: ", test_processed.shape)

Processed train: (20886, 70)
Processed test:  (5300, 69)


In [24]:
DROP_COLS = [
    'record_id', 'flood_risk_score',
    'place_name',
    'reason_not_good_to_live',
    'is_synthetic',
]

FEATURES = [c for c in train_processed.columns if c not in DROP_COLS]

CAT_FEATURES = [
    'district', 'landcover', 'soil_type', 'water_supply',
    'electricity', 'road_quality', 'urban_rural',
    'water_presence_flag', 'flood_occurrence_current_event',
    'is_good_to_live'
]
CAT_FEATURES = [c for c in CAT_FEATURES if c in FEATURES]

X = train_processed[FEATURES]
y = train_processed['flood_risk_score']
X_test = test_processed[FEATURES]

y_log = np.log1p(y)

print(f"Features: {len(FEATURES)}")
print(f"Cat features: {len(CAT_FEATURES)}")

Features: 65
Cat features: 10


In [25]:
from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()
X_test_encoded = X_test.copy()

encoders = {}
for col in CAT_FEATURES:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    X_test_encoded[col] = le.transform(
        X_test[col].astype(str).map(
            lambda x: x if x in le.classes_ else 'Missing'
        )
    )
    encoders[col] = le

print("Encoded for LightGBM/XGBoost")

Encoded for LightGBM/XGBoost


In [26]:
def competition_metric_proxy(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2 = r2_score(y_true, y_pred)
    return ((mae + rmse) / 2) * (1 + max(0, 1 - r2))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_lgbm = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f'\n--- Fold {fold+1} ---')

    Xtr_enc = X_encoded.iloc[tr_idx]
    Xval_enc = X_encoded.iloc[val_idx]
    Xtr_cat = X.iloc[tr_idx]
    Xval_cat = X.iloc[val_idx]
    ytr = y_log.iloc[tr_idx]
    yval = y.iloc[val_idx]
    ytr_orig = y.iloc[tr_idx]

    lgbm = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.03,
        num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=42, n_jobs=-1, verbose=-1
    )
    lgbm.fit(
        Xtr_enc, ytr,
        eval_set=[(Xval_enc, np.log1p(yval))],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(-1)]
    )
    oof_lgbm[val_idx] = np.expm1(lgbm.predict(Xval_enc))
    test_lgbm += np.expm1(lgbm.predict(X_test_encoded)) / 5

    xgbm = xgb.XGBRegressor(
        n_estimators=1000, learning_rate=0.03,
        max_depth=6, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgbm.fit(
        Xtr_enc, ytr,
        eval_set=[(Xval_enc, np.log1p(yval))],
        verbose=False
    )
    oof_xgb[val_idx] = np.expm1(xgbm.predict(Xval_enc))
    test_xgb += np.expm1(xgbm.predict(X_test_encoded)) / 5

    catb = CatBoostRegressor(
        iterations=1000, learning_rate=0.03,
        depth=6, l2_leaf_reg=3,
        random_seed=42, verbose=0,
        early_stopping_rounds=50,
        loss_function='RMSE'
    )
    catb.fit(
        Xtr_cat, ytr,
        cat_features=CAT_FEATURES,
        eval_set=(Xval_cat, np.log1p(yval)),
        verbose=False
    )
    oof_cat[val_idx] = np.expm1(catb.predict(Xval_cat))
    test_cat += np.expm1(catb.predict(X_test)) / 5

    for name, oof in [('LGBM', oof_lgbm), ('XGB', oof_xgb), ('CAT', oof_cat)]:
        val_preds = np.clip(oof[val_idx], 0, 1)
        print(f'  {name}: {competition_metric_proxy(yval, val_preds):.4f}')


--- Fold 1 ---
  LGBM: 0.4139
  XGB: 0.4338
  CAT: 0.4112

--- Fold 2 ---
  LGBM: 0.4208
  XGB: 0.4373
  CAT: 0.4142

--- Fold 3 ---
  LGBM: 0.4094
  XGB: 0.4274
  CAT: 0.4056

--- Fold 4 ---
  LGBM: 0.4184
  XGB: 0.4380
  CAT: 0.4123

--- Fold 5 ---
  LGBM: 0.4161
  XGB: 0.4408
  CAT: 0.4114


In [27]:
oof_lgbm_c = np.clip(oof_lgbm, 0, 1)
oof_xgb_c = np.clip(oof_xgb, 0, 1)
oof_cat_c = np.clip(oof_cat, 0, 1)

print('=== Individual OOF Scores ===')
for name, oof in [('LightGBM', oof_lgbm_c),
                  ('XGBoost', oof_xgb_c),
                  ('CatBoost', oof_cat_c)]:
    score = competition_metric_proxy(y, oof)
    rmse = np.sqrt(mean_squared_error(y, oof))
    r2 = r2_score(y, oof)
    print(f'  {name}: metric={score:.4f}  rmse={rmse:.4f}  r2={r2:.4f}')

stacker = LinearRegression(positive=True, fit_intercept=False)
oof_stack = np.column_stack([oof_lgbm_c, oof_xgb_c, oof_cat_c])
stacker.fit(oof_stack, y)

weights = stacker.coef_ / stacker.coef_.sum()
print('\nLearned blend weights:')
print(f'  LightGBM : {weights[0]:.3f}')
print(f'  XGBoost  : {weights[1]:.3f}')
print(f'  CatBoost : {weights[2]:.3f}')

oof_blend = np.clip(stacker.predict(oof_stack), 0, 1)
print(f'\nBlended OOF score: {competition_metric_proxy(y, oof_blend):.4f}')

=== Individual OOF Scores ===
  LightGBM: metric=0.4157  rmse=0.2371  r2=0.0138
  XGBoost: metric=0.4354  rmse=0.2424  r2=-0.0311
  CatBoost: metric=0.4109  rmse=0.2359  r2=0.0239

Learned blend weights:
  LightGBM : 0.035
  XGBoost  : 0.023
  CatBoost : 0.942

Blended OOF score: 0.4082


In [28]:
import os
os.makedirs('../submissions', exist_ok=True)

test_stack = np.column_stack([
    np.clip(test_lgbm, 0, 1),
    np.clip(test_xgb, 0, 1),
    np.clip(test_cat, 0, 1)
])
final_preds = np.clip(stacker.predict(test_stack), 0, 1)

submission = pd.DataFrame({
    'record_id': test['record_id'],
    'flood_risk_score': final_preds
})
submission.to_csv('../submissions/sub_ensemble_v1.csv', index=False)

print('Submission saved')
print(submission['flood_risk_score'].describe().round(4))
print('\nSample:')
print(submission.head())

Submission saved
count    5300.0000
mean        0.4778
std         0.0383
min         0.3768
25%         0.4502
50%         0.4763
75%         0.5035
max         0.6191
Name: flood_risk_score, dtype: float64

Sample:
  record_id  flood_risk_score
0   F104559          0.453732
1   F100765          0.459553
2   F107573          0.477586
3   F110345          0.546820
4   F118850          0.490474


In [29]:
# CatBoost alone may beat the blend on LB
# since XGB is dragging it down

# Retrain CatBoost on full data
cat_full = CatBoostRegressor(
    iterations=1000, learning_rate=0.03,
    depth=6, l2_leaf_reg=3,
    random_seed=42, verbose=0,
    loss_function='RMSE'
)
cat_full.fit(X, np.log1p(y), cat_features=CAT_FEATURES, verbose=False)
cat_preds = np.clip(np.expm1(cat_full.predict(X_test)), 0, 1)

sub_cat = pd.DataFrame({
    'record_id'       : test['record_id'],
    'flood_risk_score': cat_preds
})
sub_cat.to_csv('../submissions/sub_catboost_only.csv', index=False)
print("CatBoost only submission saved")

CatBoost only submission saved


In [30]:
# Blend only CatBoost and LightGBM
oof_2model = np.column_stack([oof_lgbm_c, oof_cat_c])
stacker_2  = LinearRegression(positive=True, fit_intercept=False)
stacker_2.fit(oof_2model, y)

weights_2  = stacker_2.coef_ / stacker_2.coef_.sum()
print(f"2-model weights:")
print(f"  LightGBM : {weights_2[0]:.3f}")
print(f"  CatBoost : {weights_2[1]:.3f}")

oof_blend_2 = np.clip(stacker_2.predict(oof_2model), 0, 1)
print(f"2-model blend OOF: {competition_metric_proxy(y, oof_blend_2):.4f}")

# Test submission
test_2model = np.column_stack([
    np.clip(test_lgbm, 0, 1),
    np.clip(test_cat,  0, 1)
])
preds_2model = np.clip(stacker_2.predict(test_2model), 0, 1)

sub_2model = pd.DataFrame({
    'record_id'       : test['record_id'],
    'flood_risk_score': preds_2model
})
sub_2model.to_csv('../submissions/sub_catboost_lgbm_blend.csv', index=False)
print("2-model blend saved")

2-model weights:
  LightGBM : 0.057
  CatBoost : 0.943
2-model blend OOF: 0.4082
2-model blend saved
